In [16]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

csv_file = "california_housing.csv"

In [17]:
# data = fetch_california_housing(as_frame=True)
# df = data.frame

# # 2. Save to CSV

# df.to_csv(csv_file, index=False)
# print(f"Dataset saved to {csv_file}")

In [18]:

# 1. Load real-world dataset
df = pd.read_csv(csv_file)

print("Dataset shape:", df.shape)
print(df.head())

# Features (X) and target (y)
X = df.drop(columns="MedHouseVal")  # Median House Value is the target
y = df["MedHouseVal"]

Dataset shape: (20640, 9)
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  


In [19]:
# 2. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [20]:
# 3. Feature scaling is important for SVR
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()


# Method1: Without hypertuning

In [21]:
# Following took 3-5 minutes
# 4. Train SVR model (RBF kernel works well for regression)
# SVR works well for medium-sized datasets; for huge datasets, it can be slow.

svr = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
svr.fit(X_train_scaled, y_train_scaled)


SVR(C=100, gamma=0.1)

In [22]:
# 5. Predict and inverse transform
y_pred_scaled = svr.predict(X_test_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

# 6. Evaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nModel Performance:")
print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R-square:   {r2:.3f}")


Model Performance:
MAE:  0.374
RMSE: 0.571
R-square:   0.751


# Method2: With hypertuning

In [23]:
import time

# 4. SVR model
svr = SVR(kernel="rbf")

# 5. Hyperparameter grid
param_grid = {
    "C": [1, 10, 100],
    "epsilon": [0.01, 0.1, 0.5],
    "gamma": ["scale", 0.01, 0.1, 1]
}

# 6. GridSearchCV
grid_search = GridSearchCV(
    estimator=svr,
    param_grid=param_grid,
    scoring="r2",
    cv=3,
    verbose=2,
    n_jobs=-1
)


start_time = time.time()# Start timer

grid_search.fit(X_train_scaled, y_train_scaled)

# 7. Best parameters

end_time = time.time()# End timer
elapsed_time = (end_time - start_time)/60

# 7. Best parameters & timing
print("Best Parameters:", grid_search.best_params_)
print("Best CV R² Score:", grid_search.best_score_)
print(f"Time taken: {elapsed_time:.2f} minutes")


# 8. Evaluate on test set
best_svr = grid_search.best_estimator_
y_pred_scaled = best_svr.predict(X_test_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nTest Set Performance:")
print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R-square:   {r2:.3f}")

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best Parameters: {'C': 10, 'epsilon': 0.1, 'gamma': 'scale'}
Best CV R² Score: 0.7579058490273716
Time taken: 7.95 minutes

Test Set Performance:
MAE:  0.377
RMSE: 0.568
R-square:   0.754
